In [15]:
import torch
print("CUDA beschikbaar:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA beschikbaar: True
Device: Tesla T4


In [16]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

In [17]:
URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(URL)
df = df.dropna(subset=["smiles"]).reset_index(drop=True)

print(f"Aantal moleculen: {len(df)}")
print(f"Class balance:\n{df['p_np'].value_counts()}")

Aantal moleculen: 2050
Class balance:
p_np
1    1567
0     483
Name: count, dtype: int64


In [18]:
from sklearn.model_selection import train_test_split, StratifiedKFold

smiles = df["smiles"].tolist()
labels = df["p_np"].astype(int).tolist()

# Houd een vaste test set apart (10%, stratified). Deze gebruik je ook
# straks voor ChemBERTa zodat de vergelijking eerlijk is.
trainval_smiles, test_smiles, trainval_labels, test_labels = train_test_split(
    smiles, labels, test_size=0.1, random_state=42, stratify=labels
)

print(f"Train+Val (voor CV): {len(trainval_smiles)} | Test: {len(test_smiles)}")
print(f"Class balance train+val: {np.bincount(trainval_labels)}")
print(f"Class balance test:      {np.bincount(test_labels)}")

Train+Val (voor CV): 1845 | Test: 205
Class balance train+val: [ 435 1410]
Class balance test:      [ 48 157]


In [19]:
all_chars = set()
for smi in trainval_smiles:   # was: train_smiles
    all_chars.update(smi)

sorted_chars = sorted(all_chars)
char_to_idx = {"<PAD>": 0, "<UNK>": 1}
for i, c in enumerate(sorted_chars, start=2):
    char_to_idx[c] = i

idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(char_to_idx)
print(f"Vocab size: {vocab_size}")

Vocab size: 41


In [20]:
lengths = [len(s) for s in train_smiles]
print(f"SMILES lengtes — min: {min(lengths)}, max: {max(lengths)}, "
      f"mean: {np.mean(lengths):.1f}, 95-percentiel: {int(np.percentile(lengths, 95))}")

SMILES lengtes — min: 3, max: 400, mean: 51.8, 95-percentiel: 106


In [21]:
MAX_LENGTH = 200

def encode_smiles(smi, char_to_idx, max_length=MAX_LENGTH):
    """Zet een SMILES-string om in een lijst integers van vaste lengte."""
    # Truncate als hij te lang is
    smi = smi[:max_length]
    # Map elk character naar zijn index, onbekende → <UNK>
    ids = [char_to_idx.get(c, char_to_idx["<UNK>"]) for c in smi]
    # Pad rechts met 0 (=<PAD>) tot max_length
    ids = ids + [char_to_idx["<PAD>"]] * (max_length - len(ids))
    return ids


class SMILES_CNN_Dataset(Dataset):
    def __init__(self, smiles, labels, char_to_idx, max_length=MAX_LENGTH):
        self.smiles = smiles
        self.labels = labels
        self.char_to_idx = char_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        ids = encode_smiles(self.smiles[idx], self.char_to_idx, self.max_length)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_ds = SMILES_CNN_Dataset(train_smiles, train_labels, char_to_idx)
val_ds   = SMILES_CNN_Dataset(val_smiles,   val_labels,   char_to_idx)
test_ds  = SMILES_CNN_Dataset(test_smiles,  test_labels,  char_to_idx)

# Even checken
print(f"Train dataset size: {len(train_ds)}")
print(f"Eerste sample shape: {train_ds[0]['input_ids'].shape}")
print(f"Eerste 30 tokens van eerste sample: {train_ds[0]['input_ids'][:30]}")

Train dataset size: 1660
Eerste sample shape: torch.Size([200])
Eerste 30 tokens van eerste sample: tensor([35, 11, 35, 35, 35, 12, 35,  4, 35,  4, 23,  4, 27, 31, 23, 21, 21, 25,
        33,  4, 23, 23,  5, 35, 13, 35, 35, 35, 35, 35])


In [22]:
class SMILES_CNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=64,
                 kernel_sizes=(3, 5, 7), dropout=0.5, num_classes=2,
                 class_weights=None):   # NIEUW
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

        # NIEUW: bewaar class weights als buffer (gaat automatisch mee naar GPU)
        if class_weights is not None:
            self.register_buffer(
                "class_weights",
                torch.tensor(class_weights, dtype=torch.float32)
            )
        else:
            self.class_weights = None

    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        x = x.permute(0, 2, 1)

        conv_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x))
            p = F.max_pool1d(c, c.size(2)).squeeze(2)
            conv_outputs.append(p)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        logits = self.fc(x)

        loss = None
        if labels is not None:
            # NIEUW: weight= toegevoegd
            loss = F.cross_entropy(logits, labels, weight=self.class_weights)

        return {"loss": loss, "logits": logits}

In [23]:
from torch.utils.data import DataLoader

# DataLoaders: zorgen voor batching en (voor train) shuffling
BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)


def evaluate(model, loader, device):
    """Evalueer model op een loader, geeft loss/accuracy/AUC terug."""
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            out = model(input_ids, labels=labels)
            total_loss += out["loss"].item() * input_ids.size(0)
            all_logits.append(out["logits"].cpu())
            all_labels.append(labels.cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels).numpy()
    preds = logits.argmax(dim=1).numpy()
    probs = torch.softmax(logits, dim=1)[:, 1].numpy()

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(labels, preds),
        "roc_auc":  roc_auc_score(labels, probs),
    }


def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, weight_decay=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_auc = 0.0
    best_state = None
    history = []

    print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>9} | {'Val Acc':>7} | {'Val AUC':>7}")
    print("-" * 55)

    for epoch in range(1, epochs + 1):
        # ---- Training ----
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            out = model(input_ids, labels=labels)
            out["loss"].backward()
            optimizer.step()

            epoch_loss += out["loss"].item() * input_ids.size(0)

        train_loss = epoch_loss / len(train_loader.dataset)

        # ---- Validation ----
        val_metrics = evaluate(model, val_loader, device)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics,
        })

        print(f"{epoch:>5} | {train_loss:>10.4f} | {val_metrics['loss']:>9.4f} "
              f"| {val_metrics['accuracy']:>7.4f} | {val_metrics['roc_auc']:>7.4f}")

        # Sla het beste model op (op basis van val AUC)
        if val_metrics["roc_auc"] > best_val_auc:
            best_val_auc = val_metrics["roc_auc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Laad het beste model terug
    model.load_state_dict(best_state)
    print(f"\nBeste val AUC: {best_val_auc:.4f}")
    return history

In [24]:
from sklearn.utils.class_weight import compute_class_weight

N_FOLDS = 5
EPOCHS = 30
BATCH_SIZE = 32

torch.manual_seed(42)
np.random.seed(42)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []
fold_models = []  # bewaar getrainde modellen per fold

trainval_smiles_arr = np.array(trainval_smiles)
trainval_labels_arr = np.array(trainval_labels)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(trainval_smiles_arr, trainval_labels_arr), start=1
):
    print(f"\n========== FOLD {fold}/{N_FOLDS} ==========")

    # Split deze fold
    fold_train_smiles = trainval_smiles_arr[train_idx].tolist()
    fold_train_labels = trainval_labels_arr[train_idx].tolist()
    fold_val_smiles   = trainval_smiles_arr[val_idx].tolist()
    fold_val_labels   = trainval_labels_arr[val_idx].tolist()

    # Class weights per fold berekenen (op basis van train-portie alleen)
    cw = compute_class_weight(
        "balanced",
        classes=np.array([0, 1]),
        y=np.array(fold_train_labels)
    )
    print(f"Class weights deze fold: {cw}")

    # Datasets + loaders
    fold_train_ds = SMILES_CNN_Dataset(fold_train_smiles, fold_train_labels, char_to_idx)
    fold_val_ds   = SMILES_CNN_Dataset(fold_val_smiles,   fold_val_labels,   char_to_idx)
    fold_train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE, shuffle=True)
    fold_val_loader   = DataLoader(fold_val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    # Nieuw model per fold (anders leer je verder op vorige fold!)
    fold_model = SMILES_CNN(vocab_size=vocab_size, class_weights=cw).to(device)

    # Train (gebruik je bestaande train_model functie)
    history = train_model(fold_model, fold_train_loader, fold_val_loader, epochs=EPOCHS)

    # Sla beste val-metrics op
    best_val = max(history, key=lambda h: h["roc_auc"])
    fold_results.append(best_val)
    fold_models.append(fold_model)

# Samenvatting
print("\n========== CV RESULTATEN ==========")
val_aucs = [r["roc_auc"] for r in fold_results]
val_accs = [r["accuracy"] for r in fold_results]
print(f"Val ROC-AUC per fold: {[f'{a:.4f}' for a in val_aucs]}")
print(f"Mean ± std ROC-AUC:   {np.mean(val_aucs):.4f} ± {np.std(val_aucs):.4f}")
print(f"Mean ± std Accuracy:  {np.mean(val_accs):.4f} ± {np.std(val_accs):.4f}")


========== FOLD 1/5 ==========
Class weights deze fold: [2.12068966 0.65425532]
Epoch | Train Loss |  Val Loss | Val Acc | Val AUC
-------------------------------------------------------
    1 |     0.6010 |    0.4863 |  0.7182 |  0.9214
    2 |     0.4233 |    0.3293 |  0.8645 |  0.9374
    3 |     0.3662 |    0.3206 |  0.8591 |  0.9359
    4 |     0.3323 |    0.4103 |  0.7642 |  0.9364
    5 |     0.3045 |    0.3396 |  0.8103 |  0.9395
    6 |     0.2645 |    0.3127 |  0.8482 |  0.9405
    7 |     0.2443 |    0.3092 |  0.8401 |  0.9427
    8 |     0.2698 |    0.2971 |  0.8347 |  0.9448
    9 |     0.2353 |    0.2924 |  0.8428 |  0.9479
   10 |     0.2386 |    0.2974 |  0.8482 |  0.9455
   11 |     0.2258 |    0.3116 |  0.8482 |  0.9427
   12 |     0.2150 |    0.3018 |  0.8509 |  0.9499
   13 |     0.1814 |    0.3086 |  0.8672 |  0.9480
   14 |     0.1692 |    0.3103 |  0.8482 |  0.9475
   15 |     0.1892 |    0.3158 |  0.8509 |  0.9446
   16 |     0.1809 |    0.2975 |  0.8591 |  0.9

In [25]:
test_ds = SMILES_CNN_Dataset(test_smiles, test_labels, char_to_idx)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

all_probs_per_fold = []
for m in fold_models:
    m.eval()
    probs = []
    with torch.no_grad():
        for batch in test_loader:
            out = m(batch["input_ids"].to(device))
            probs.append(torch.softmax(out["logits"], dim=1)[:, 1].cpu().numpy())
    all_probs_per_fold.append(np.concatenate(probs))

# Middel de probabilities
mean_probs = np.mean(all_probs_per_fold, axis=0)
y_pred = (mean_probs > 0.5).astype(int)
y_true = np.array(test_labels)

print("=== Test resultaten (ensemble van 5 folds) ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_true, mean_probs):.4f}")
print(f"\nConfusion matrix:\n{confusion_matrix(y_true, y_pred)}")
print(f"\n{classification_report(y_true, y_pred, target_names=['geen BBB', 'wel BBB'])}")

=== Test resultaten (ensemble van 5 folds) ===
Accuracy: 0.9220
ROC-AUC:  0.9723

Confusion matrix:
[[ 44   4]
 [ 12 145]]

              precision    recall  f1-score   support

    geen BBB       0.79      0.92      0.85        48
     wel BBB       0.97      0.92      0.95       157

    accuracy                           0.92       205
   macro avg       0.88      0.92      0.90       205
weighted avg       0.93      0.92      0.92       205

